# Case study: product clusters from review statistics

*Session 11, blocks 2 and 3 practice. Author: course team, licence CC-BY-4.0.*

**Part A (block 2).** Describe each product by statistics of its reviews, and compare k-means, hierarchical clustering and DBSCAN on these product-level features.

**Part B (block 3).** Visualise the products with PCA and test whether the product cluster improves a gradient-boosting model of review sentiment (the tree-based model of Session 10).

**Avoiding leakage.** The product statistics include the ratings of the reviews. If we computed them from all reviews and then predicted the sentiment of the same reviews, the target would leak into the features (Session 9). We therefore build the product features from reviews **up to 2019**, train the sentiment model on reviews from **2020** and test it on reviews from **2021**.

Theory: [02-hierarchical-dbscan-and-evaluation.md](../theory/02-hierarchical-dbscan-and-evaluation.md), [03-dimensionality-reduction.md](../theory/03-dimensionality-reduction.md), [04-anomaly-detection-and-unsupervised-features.md](../theory/04-anomaly-detection-and-unsupervised-features.md).

In [ ]:
from pathlib import Path

# find the repository root (the folder that contains case-study/), wherever the notebook is started
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").exists())
DATA = ROOT / "case-study" / "data"

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.cluster import DBSCAN, AgglomerativeClustering, KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import adjusted_rand_score, f1_score, silhouette_score
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)
cols = ["parent_asin", "rating", "label", "helpful_vote", "verified_purchase", "n_images", "text", "date"]
reviews = pd.read_parquet(DATA / "train.parquet", columns=cols)
reviews["length"] = reviews["text"].fillna("").str.len()
reviews["date"].min(), reviews["date"].max(), len(reviews)

# Part A: product features and three clustering methods

## A1. Product-level features (reviews up to 2019)

Only products with at least 20 reviews before 2020 are kept, so that the statistics are not dominated by chance.

In [ ]:
early = reviews[reviews["date"] < "2020-01-01"].assign(
    negative=lambda d: d["rating"] <= 2,
    has_image=lambda d: d["n_images"] > 0,
)
products = early.groupby("parent_asin").agg(
    n_reviews=("rating", "size"),
    mean_rating=("rating", "mean"),
    sd_rating=("rating", "std"),
    share_negative=("negative", "mean"),
    share_verified=("verified_purchase", "mean"),
    mean_helpful=("helpful_vote", "mean"),
    median_length=("length", "median"),
    share_images=("has_image", "mean"),
    first=("date", "min"),
    last=("date", "max"),
)
products["years_active"] = (products["last"] - products["first"]).dt.days / 365.25
products = products.drop(columns=["first", "last"]).query("n_reviews >= 20")
print(len(products), "products")
products.describe().round(2).T

In [ ]:
# log-transform the right-skewed columns, then standardise
features = products.copy()
for col in ["n_reviews", "mean_helpful", "median_length"]:
    features[col] = np.log1p(features[col])
Z = StandardScaler().fit_transform(features)
Z.shape

## A2. k-means: choose k

In [ ]:
rows = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(Z)
    rows.append({"k": k, "inertia": round(km.inertia_), "silhouette": round(silhouette_score(Z, km.labels_), 3)})
pd.DataFrame(rows)

The silhouette stays around 0.15–0.20 for all k: the products do not form well-separated groups. We continue with k = 4 as an interpretable summary.

In [ ]:
K = 4
km = KMeans(n_clusters=K, n_init=10, random_state=0).fit(Z)
products["kmeans"] = km.labels_

## A3. Hierarchical clustering (Ward)

In [ ]:
tree = linkage(Z, method="ward")
fig, ax = plt.subplots(figsize=(10, 3.5))
dendrogram(tree, truncate_mode="lastp", p=25, ax=ax, no_labels=True, color_threshold=tree[-K + 1, 2])
ax.set(title="Ward dendrogram of 2,413 products (top 25 merges)", ylabel="merge height")
plt.tight_layout()

ward = AgglomerativeClustering(n_clusters=K, linkage="ward").fit(Z)
products["ward"] = ward.labels_

## A4. DBSCAN

Choose eps from the distances to the 10th nearest neighbour.

In [ ]:
dist, _ = NearestNeighbors(n_neighbors=10).fit(Z).kneighbors(Z)
kdist = np.sort(dist[:, -1])
plt.figure(figsize=(6, 3))
plt.plot(kdist)
plt.gca().set(xlabel="products sorted by distance", ylabel="distance to 10th neighbour", title="k-distance plot")
plt.tight_layout()

for eps in (1.0, 1.5, 2.0):
    labels = DBSCAN(eps=eps, min_samples=10).fit(Z).labels_
    print(eps, pd.Series(labels).value_counts().to_dict())
products["dbscan"] = DBSCAN(eps=1.5, min_samples=10).fit(Z).labels_

## A5. Compare the three methods

In [ ]:
summary = []
for method in ["kmeans", "ward", "dbscan"]:
    labels = products[method]
    clustered = labels != -1                       # DBSCAN noise is excluded from its silhouette
    n_clusters = labels[clustered].nunique()
    sil = silhouette_score(Z[clustered], labels[clustered]) if n_clusters > 1 else np.nan
    summary.append({"method": method, "clusters": n_clusters, "noise": int((~clustered).sum()),
                    "silhouette": round(sil, 3), "sizes": labels.value_counts().sort_index().to_dict()})
print(pd.DataFrame(summary).to_string(index=False))
print("ARI k-means vs Ward:", round(adjusted_rand_score(products["kmeans"], products["ward"]), 2))

In [ ]:
# profiles on the original scale
show = ["n_reviews", "mean_rating", "share_negative", "share_verified", "median_length", "years_active"]
products.groupby("kmeans")[show].median().round(2).assign(size=products["kmeans"].value_counts().sort_index())

In [ ]:
# what makes the DBSCAN noise products unusual? compare medians
products.groupby(products["dbscan"] == -1)[show].median().round(2).rename(index={False: "core", True: "noise"})

**Questions for Part A.**
1. Do k-means and Ward agree? Which clusters are found by both (compare the profiles)?
2. DBSCAN finds one large cluster. What does this say about the structure of the product data?
3. Give each k-means cluster a short name based on its profile.

# Part B: PCA picture and clusters as features

## B1. PCA

In [ ]:
pca = PCA().fit(Z)
scores = pca.transform(Z)
print("explained variance ratio:", pca.explained_variance_ratio_.round(3))
loadings = pd.DataFrame(pca.components_[:2].T, index=features.columns, columns=["PC1", "PC2"]).round(2)
loadings

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5.5))
for k in range(K):
    m = products["kmeans"].to_numpy() == k
    ax.scatter(scores[m, 0], scores[m, 1], s=6, alpha=0.6, label=f"cluster {k}")
for name, (x, y) in loadings.iterrows():                 # loadings as arrows (biplot)
    ax.annotate("", xy=(4 * x, 4 * y), xytext=(0, 0), arrowprops=dict(arrowstyle="->"))
    ax.text(4.4 * x, 4.4 * y, name, fontsize=8, ha="center")
ratio = pca.explained_variance_ratio_
ax.set(xlabel=f"PC1 ({ratio[0]:.0%})", ylabel=f"PC2 ({ratio[1]:.0%})", title="Products in the first two principal components")
ax.legend(markerscale=3)
plt.tight_layout()

**Question.** How would you name PC1 and PC2 from the loadings? Where do the clusters lie in the picture?

## B2. Does the product cluster help a sentiment model?

Review-level features as in the leaderboard baseline (length, punctuation, negations, verified purchase, helpful votes, images), with and without the product's cluster, and with the raw product statistics for comparison. Model: `HistGradientBoostingClassifier` with balanced class weights; metric: macro-F1. Training reviews from 2020, test reviews from 2021, only products that have a cluster.

In [ ]:
late = reviews[(reviews["date"] >= "2020-01-01") & reviews["parent_asin"].isin(products.index)].copy()
text = late["text"].fillna("")
late["log_length"] = np.log1p(late["length"])
late["n_exclaim"] = text.str.count("!")
late["n_question"] = text.str.count(r"\?")
late["n_negation"] = text.str.lower().str.count(r"\b(?:not|no|never|don't|didn't|doesn't|isn't|wasn't)\b")
late["verified"] = late["verified_purchase"].astype(int)
base = ["log_length", "n_exclaim", "n_question", "n_negation", "verified", "helpful_vote", "n_images"]

late = late.join(products[["kmeans"]], on="parent_asin").join(features.add_prefix("p_"), on="parent_asin")
product_stats = [c for c in late.columns if c.startswith("p_")]
train, test = late[late["date"] < "2021-01-01"], late[late["date"] >= "2021-01-01"]
print(len(train), "training reviews,", len(test), "test reviews")
test["label"].value_counts(normalize=True).round(3)

In [ ]:
variants = {
    "review features only": (base, None),
    "+ product cluster": (base + ["kmeans"], ["kmeans"]),
    "+ product statistics": (base + product_stats, None),
}
predictions = {}
for name, (cols_, categorical) in variants.items():
    model = HistGradientBoostingClassifier(random_state=0, class_weight="balanced",
                                           categorical_features=categorical)
    model.fit(train[cols_], train["label"])
    predictions[name] = model.predict(test[cols_])
    print(f"{name:22s} macro-F1 {f1_score(test['label'], predictions[name], average='macro'):.3f}")

Is the difference larger than chance? A paired bootstrap of the test reviews gives an interval for the gain of the cluster feature.

In [ ]:
rng = np.random.default_rng(0)
y_true = test["label"].to_numpy()
a, b = predictions["review features only"], predictions["+ product cluster"]
gains = []
for _ in range(300):
    idx = rng.integers(0, len(y_true), len(y_true))
    gains.append(f1_score(y_true[idx], b[idx], average="macro") - f1_score(y_true[idx], a[idx], average="macro"))
print("gain in macro-F1, 95 % bootstrap interval:", np.quantile(gains, [0.025, 0.975]).round(3))

**Questions for Part B.**
1. Does the cluster improve the model? Does it help more or less than the raw product statistics, and why might that be?
2. Why were the product features built from reviews before 2020 only? What would happen to the test score if they were built from all training reviews?
3. Products without 20 reviews before 2020 are not in the test. How would you handle new products in a deployed model?

## Exercises

1. Replace the cluster label by the distances to the four k-means centroids (`km.transform(Z)`). Does the model improve?
2. Use the first three principal component scores of the products as features instead of the cluster.
3. Score the products with `IsolationForest` and inspect the ten most unusual ones (look up their titles in `products.parquet`). Are they errors or interesting cases?

In [ ]:
# Exercise 1: your code here
